## Gold Comparison & Evaluation (Deliverable 1.3.4)

This notebook performs the final comparison between the baseline model and the three-stage cascade. It evaluates whether the cascade meaningfully improves alert quality by reducing false positives and noisy alerts while preserving useful anomaly sensitivity.

**Purpose:**  
To combine the alert outputs from the Baseline Modeling and Cascade Modeling notebooks, compute evaluation metrics, perform paired statistical testing, and generate the comparative visualizations required for Section C.

**Key Goals:**

- Load baseline and cascade alert outputs.
- Align alerts into comparable time windows for paired evaluation.
- Compute alert-volume metrics, false-positive rates, normal-period alerts, and anomaly responsiveness.
- Perform the planned statistical significance test (paired/Wilcoxon) to quantify model differences.
- Produce comparison tables, charts, and anomaly overlays for Section C.6.
- Summarize both practical and statistical significance findings based on the project’s research question.

**Relevance to Section C:**  
This notebook fulfills the analytical requirements of C.2, C.4, C.5, and C.6 by generating:  
- The model comparison metrics,  
- The statistical significance results,  
- The practical significance interpretation, and  
- All visual communication elements needed for the final report.

This completes the Gold layer and provides the definitive evidence used to answer the project’s research question.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

#import os
#import glob

from pathlib import Path
import yaml
import re

import logging
import wandb

import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

import joblib 

from sklearn.model_selection import train_test_split, KFold

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, RobustScaler

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, roc_auc_score, average_precision_score

from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor

import pyarrow.parquet as pq
import pyarrow as pa


import hashlib


# Custom Utilities Module
from utils.paths import get_paths
from utils.file_io import load_data, save_data, save_json, load_json
from utils.eda_logging import profile_dataframe
from utils.logging_setup import configure_logging
from utils.wandb_utils import finalize_wandb_stage

# Ledger 
from utils.ledger import Ledger

# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def log_gold_paths(paths, logger: logging.Logger) -> None:
    logger.info("Project Root Path Loaded: %s", paths.root)
    logger.info("Project Logging Path Loaded: %s", paths.logs)
    logger.info("Project Artifacts Path Loaded: %s", paths.artifacts)
    logger.info("Project Notebooks Path Loaded %s", paths.notebooks)
    logger.info("Project Data Path Loaded: %s", paths.data)
    logger.info("Data Bronze Path Loaded: %s", paths.data_bronze)
    logger.info("Data Bronze Training Path Loaded: %s", paths.data_bronze_train)
    logger.info("Data Bronze Testing Path Loaded: %s", paths.data_bronze_test)
    logger.info("Data Silver Path Loaded: %s", paths.data_silver)
    logger.info("Data Silver Training Path Loaded: %s", paths.data_silver_train)
    logger.info("Data Silver Testing Path Loaded: %s", paths.data_silver_test)
    logger.info("Data Gold Path Loaded: %s", paths.data_gold)

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Configurables

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Stage Details
STAGE = "gold"
LAYER_NAME = "gold"
GOLD_VERSION = "gold__001"
RECIPE_ID = "gold_modeling__v001_model_comparision"


DATASET_NAME_CONFIG = "pump"
DATASET_NAME = str(DATASET_NAME_CONFIG).strip().lower()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Weights and Biases
WANDB_PROJECT = "capstone"
WANDB_ENTITY = "dcoo230-western-governors-university"
WANDB_RUN_NAME = f"{GOLD_VERSION}"


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# File names
GOLD_FILE_NAME = f"{DATASET_NAME}__gold__preprocessed.parquet"

STAGE1_FEATURES_FILE_NAME = f"{DATASET_NAME}__gold__stage1_features.json"
STAGE2_FEATURES_FILE_NAME = f"{DATASET_NAME}__gold__stage2_features.json"
STAGE3_PRIMARY_FILE_NAME = f"{DATASET_NAME}__gold__stage3_primary_rule_sensors.json"
STAGE3_SECONDARY_FILE_NAME = f"{DATASET_NAME}__gold__stage3_secondary_rule_sensors.json"

CASCADE_RESULTS_FILE_NAME = f"{DATASET_NAME}__gold__cascade_results.csv"

COMPARISON_FILE_NAME = f"{DATASET_NAME}__gold__comparison__baseline_vs_cascade.csv"


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Split configuration
TRAIN_FRACTION = 0.70
RANDOM_SEED = 42

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Scaling posture
USE_ROBUST_SCALER = False

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Cascade thresholding
STAGE1_THRESHOLD_PERCENTILE = 95.0   # broader / more sensitive
STAGE2_THRESHOLD_PERCENTILE = 98.5   # narrower / stricter

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Isolation Forest size
STAGE1_ESTIMATOR_COUNT = 200
STAGE2_ESTIMATOR_COUNT = 300

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 







In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Paths Setup

# Get Path's Object
paths = get_paths()

# Gold
GOLD_DATA_PATH = paths.data_gold / GOLD_FILE_NAME
GOLD_ARTIFACTS_PATH = paths.artifacts / "gold" / DATASET_NAME

BASELINE_RESULTS_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__baseline_results.csv"
BASELINE_SUMMARY_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__baseline_summary.json"

CASCADE_RESULTS_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__cascade_results.csv"
CASCADE_SUMMARY_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__cascade_summary.json"

BASELINE_VS_CASCADE_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__baseline_vs_cascade.csv"
BASELINE_VS_CASCADE_SUMMARY_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__baseline_vs_cascade_summary.json"

# Logs
LOGS_PATH = paths.logs

# Path Failsafes
GOLD_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)



In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Logging Setup

# Create gold log path 
gold_log_path = paths.logs / "gold_model_comparision.log"

# Initial Logger
configure_logging(
    "capstone",
    gold_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.gold")

# Log load and initiation
logger.info("Gold Modeling stage starting")

# Log paths loads
log_gold_paths(paths, logger)


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="gold_model_comparision",
    config={
    "gold_version": GOLD_VERSION,
    "dataset": DATASET_NAME,
    "stage": STAGE,
    "baseline_results_path": str(BASELINE_RESULTS_PATH),
    "baseline_summary_path": str(BASELINE_SUMMARY_PATH),
    "cascade_results_path": str(CASCADE_RESULTS_PATH),
    "cascade_summary_path": str(CASCADE_SUMMARY_PATH),
    },
)
logger.info("W&B initialized: %s", wandb.run.name)


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
baseline_results = pd.read_csv(BASELINE_RESULTS_PATH)
baseline_summary = load_json_file(BASELINE_SUMMARY_PATH)

cascade_results = pd.read_csv(CASCADE_RESULTS_PATH)
cascade_summary = load_json_file(CASCADE_SUMMARY_PATH)

ledger.add(
    kind="step",
    step="load_comparison_inputs",
    message="Loaded saved baseline and cascade outputs for final comparison.",
    data={
        "baseline_results_path": str(BASELINE_RESULTS_PATH),
        "baseline_summary_path": str(BASELINE_SUMMARY_PATH),
        "cascade_results_path": str(CASCADE_RESULTS_PATH),
        "cascade_summary_path": str(CASCADE_SUMMARY_PATH),
        "baseline_result_rows": int(len(baseline_results)),
        "cascade_result_rows": int(len(cascade_results)),
    },
    logger=logger,
)

baseline_results.head(3)

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
baseline_alert_count = int(baseline_summary.get("alert_count_all_rows", 0))

stage1_alert_count = int(cascade_summary.get("stage1_alert_count_all_rows", 0))
stage2_alert_count = int(cascade_summary.get("stage2_alert_count_all_rows", 0))
final_cascade_alert_count = int(cascade_summary.get("final_cascade_alert_count_all_rows", 0))

alert_reduction_count = baseline_alert_count - final_cascade_alert_count
alert_reduction_ratio = alert_reduction_count / max(baseline_alert_count, 1)

baseline_metrics = baseline_summary.get("baseline_metrics", {})
cascade_metrics = cascade_summary.get("cascade_metrics", {})

comparison_rows = [
    {
        "Model": "Baseline IsolationForest",
        "Test_Alerts": baseline_alert_count,
        "Precision": baseline_metrics.get("precision"),
        "Recall": baseline_metrics.get("recall"),
        "F1": baseline_metrics.get("f1"),
    },
    {
        "Model": "3-Stage Cascade",
        "Test_Alerts": final_cascade_alert_count,
        "Precision": cascade_metrics.get("precision"),
        "Recall": cascade_metrics.get("recall"),
        "F1": cascade_metrics.get("f1"),
    },
]

comparison_table = pd.DataFrame(comparison_rows)

comparison_summary = {
    "baseline_alert_count": baseline_alert_count,
    "stage1_alert_count": stage1_alert_count,
    "stage2_alert_count": stage2_alert_count,
    "final_cascade_alert_count": final_cascade_alert_count,
    "alert_reduction_count": int(alert_reduction_count),
    "alert_reduction_ratio": float(alert_reduction_ratio),
    "baseline_precision": baseline_metrics.get("precision"),
    "baseline_recall": baseline_metrics.get("recall"),
    "baseline_f1": baseline_metrics.get("f1"),
    "cascade_precision": cascade_metrics.get("precision"),
    "cascade_recall": cascade_metrics.get("recall"),
    "cascade_f1": cascade_metrics.get("f1"),
}

ledger.add(
    kind="step",
    step="compare_baseline_vs_cascade",
    message="Compared baseline output against final cascade output.",
    data=comparison_summary,
    logger=logger,
)


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
try:
    GOLD_ARTIFACTS_PATH
    out_dir = Path(GOLD_ARTIFACTS_PATH)
except NameError:
    out_dir = Path("./artifacts/gold_modeling")

out_dir.mkdir(parents=True, exist_ok=True)
comparison_path = out_dir / "pump__gold__comparison__baseline_vs_cascade.csv"
comparison_table.to_csv(comparison_path, index=False)

print(f"Saved comparison table to: {comparison_path}")


In [ ]:
styled = (
    comparison_table
    .style.format({
        "Test_Alerts": "{:,}",
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "F1": "{:.3f}",
    })
    .set_caption("Gold Model Performance Comparison: Baseline vs Cascade")
)

display(HTML(styled.to_html()))

In [ ]:
metrics_to_plot = ["Precision", "Recall", "F1"]

models = comparison_table["Model"].tolist()
n_models = len(models)
n_metrics = len(metrics_to_plot)

values = []
for metric in metrics_to_plot:
    values.append(comparison_table[metric].values)

values = np.array(values)  # shape: (n_metrics, n_models)

x = np.arange(n_metrics)  # positions for metrics
width = 0.35  # bar width

fig, ax = plt.subplots(figsize=(8, 5))

for i, model in enumerate(models):
    offset = (i - (n_models - 1) / 2) * width
    ax.bar(x + offset, values[:, i], width=width, label=model)

ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.0)
ax.set_title("Gold Model Performance: Precision, Recall, F1")
ax.legend()
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

# Save the bar chart as an artifact
plot_path = out_dir / "pump__gold__comparison__metrics_barplot.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved comparison plot to: {plot_path}")

# Check for wandb
if "wandb_run" in globals() and wandb_run is not None:
    try:
        import wandb

        # Log the PNG as an image
        wandb_run.log({
            "gold/comparison_plot": wandb.Image(str(plot_path))
        })

        # Also log the comparison table as a W&B Table
        wandb_run.log({
            "gold/model_comparison_table": wandb.Table(dataframe=comparison_table)
        })

        print("W&B: logged comparison plot and table.")
    except Exception as e:
        print(f"W&B logging skipped due to error: {e}")



In [ ]:
comparison_table.to_csv(BASELINE_VS_CASCADE_PATH, index=False)
save_json_file(comparison_summary, BASELINE_VS_CASCADE_SUMMARY_PATH)

wandb.save(str(BASELINE_VS_CASCADE_PATH))
wandb.save(str(BASELINE_VS_CASCADE_SUMMARY_PATH))

ledger.add(
    kind="step",
    step="save_comparison_outputs",
    message="Saved final baseline versus cascade comparison outputs.",
    data={
        "comparison_csv": str(BASELINE_VS_CASCADE_PATH),
        "comparison_summary_json": str(BASELINE_VS_CASCADE_SUMMARY_PATH),
        "comparison_rows": int(len(comparison_table)),
    },
    logger=logger,
)

In [ ]:
ledger.add(
    kind="step",
    step="finalize_comparison",
    message="Gold comparison notebook complete.",
    data={
        "comparison_csv": str(BASELINE_VS_CASCADE_PATH),
        "comparison_summary_json": str(BASELINE_VS_CASCADE_SUMMARY_PATH),
        "comparison_summary": comparison_summary,
    },
    logger=logger,
)

comparison_ledger_path = GOLD_ARTIFACTS_PATH / f"ledger__{DATASET_NAME}__gold_comparison.json"
ledger.write_json(comparison_ledger_path)

wandb.save(str(comparison_ledger_path))
wandb_run.finish()